make a csv file using labels from existing csv file. drop duplicate sounds using hashing

In [1]:
import pandas as pd
import os
from pathlib import Path
import librosa
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import hashlib

In [2]:
# get file paths
FOLDER_PATH = "/Users/cheoso/Documents/samples/Packs/"
file_paths = []
total = 0

for root, dirs, files in os.walk(FOLDER_PATH):
    for file in files:
        path = Path(root) / file
        path_lower = str(path).lower()
        if path_lower.endswith(('.wav', '.aif', '.aiff')):
          file_paths.append(str(path))

          total += path.stat().st_size

print(f"Total of {total} bytes.")

Total of 90104385888 bytes.


In [14]:
failed_files = []
divided_by_zero = []

def process_audio(audio, sr):
    rs = librosa.resample(audio, orig_sr=sr, target_sr=44100)
    if rs.max() == 0:
        divided_by_zero.append(sr)
    else:
        return rs / abs(rs.max())
    return rs

def hash_waveform_with_id(row):
    waveform = row
    waveform_hash = hashlib.sha256(waveform.tobytes()).hexdigest()
    return waveform_hash

def extract_metadata(index, file):
    """Load an audio file and extract its waveform, sample rate, and label."""
    try:
        audio_waveform, sr = librosa.load(file, sr=44100)  # Load without resampling
        audio = process_audio(audio_waveform, sr)
        audio = hash_waveform_with_id(audio)
        file = str(file)
        #time_duration = len(audio_waveform) / sr
        file_name = file.split('/')[-1]
        return [index, file, file_name, audio]
    except Exception as e:
        failed_files.append(file)
        return None

# Use parallel processing to load audio files
# add tqdm counter
def get_audio_and_metadata(files):
    meta_results = []

    with ThreadPoolExecutor() as executor:
        results = tqdm(executor.map(lambda p: extract_metadata(*p), enumerate(files)), total=len(files))
        
        for res in results:
            if res is not None:
                meta_data = res
                meta_results.append(meta_data)
    
    return meta_results


metadataResults = get_audio_and_metadata(file_paths)

  2%|▏         | 1281/60312 [00:02<01:40, 589.16it/s]/var/folders/h0/pzm07l6x5n7181rc09xppl940000gn/T/ipykernel_86794/1695125903.py:20: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_waveform, sr = librosa.load(file, sr=44100)  # Load without resampling
/Users/cheoso/ai_projects/load_audio/venv/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
100%|██████████| 60312/60312 [01:32<00:00, 654.99it/s] 


In [15]:
failed_len = len(failed_files)
print(failed_len)
print(len(divided_by_zero))

639
12


In [16]:
print(len(file_paths))

60312


In [17]:
print(len(metadataResults))

59673


In [18]:
# Convert results to DataFrame
df_md = pd.DataFrame(metadataResults, columns=["id", "file_path", "file_name", "audio_hash"])

In [20]:
# load old csv file with categorizations
reference_df = pd.read_csv('ref_2.csv')

# only keep data that has been classified
filtered_results = reference_df[reference_df['group'].notna()]
filtered_results['is_used'] = True

/var/folders/h0/pzm07l6x5n7181rc09xppl940000gn/T/ipykernel_86794/2009694902.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_results['is_used'] = True


In [21]:
# split the path and make it all lower case to match the paths in filtered results
df_md['file_path'] = df_md['file_path'].apply(lambda x: x.lower().split('/users/cheoso/documents/samples/packs/')[1])

# join data together. adds the is used column
joined_df = df_md.merge(filtered_results, on='audio_hash', how='left')
joined_df.drop(columns=[col for col in joined_df.columns if '_y' in col], inplace=True)
joined_df.rename(columns={col: col.split('_x')[0] for col in joined_df.columns if '_x' in col}, inplace=True)

# fill in missing values
joined_df['is_used'].astype(bool)
joined_df.loc[joined_df['is_used'].isna(), 'is_used'] = False

In [22]:
# sort df ensuring that used files are first, drop any duplicates while keeping the first listed audio 
# file (ensures that any duplicates that are in use are kept while all other are deleted)
joined_df = joined_df.sort_values(by='is_used', ascending=False).drop_duplicates(subset='audio_hash', keep='first', inplace=False)

In [28]:
try:
    joined_df.drop(columns=['Unnamed: 52'], inplace=True)
except Exception as e:
    print(e)

assert len(set(df_md['audio_hash'])) == len(joined_df)
joined_df.drop(columns=['is_used'], inplace=True)
joined_df.sort_values(by='file_path').to_csv("audio_metadata_testing_filt_with_hashes_2.csv", header=True, index=False, encoding='utf-8-sig')

"['Unnamed: 52'] not found in axis"
